# Customer Analytics -- RFM Segmentation, CLTV Modeling & Behavioral Clustering

**Author:** Mehmet Isik | **Date:** 2026-03 | **Kernel:** Python 3.11

---

## 1. Introduction

Understanding guest behavior is the cornerstone of modern hospitality revenue management.
This notebook provides an end-to-end customer analytics pipeline applied to a hotel group
operating across multiple properties.

**Dataset at a glance:**

| Metric | Value |
|--------|-------|
| Unique customers | ~12 000 |
| Transactions | ~53 000 |
| Date range | 2024-01 to 2025-06 |
| Channels | OTA, Direct, Travel Agent, Corporate |

**What we cover:**

1. **RFM Analysis** -- Recency / Frequency / Monetary scoring and segment assignment
2. **CLTV Modeling** -- BG-NBD + Gamma-Gamma probabilistic lifetime value
3. **K-Means Clustering** -- Behavioral segmentation with radar profiling
4. **Business Recommendations** -- Actionable takeaways per segment

All visualizations use **Plotly** for Kaggle-friendly interactivity.

---
## 2. Setup & Imports

In [1]:
!pip install lifetimes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.2/584.2 kB 15.9 MB/s eta 0:00:00


In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from datetime import timedelta

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# CLTV modeling
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data

# Clustering
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)

print('All libraries loaded successfully.')

All libraries loaded successfully.


---
## 3. Data Loading & Initial Exploration

In [3]:
# ----- Adjust paths for Kaggle vs local -----
# On Kaggle, upload the CSVs as a dataset and use:
#   Local: DATA_DIR = '../data/synthetic/'
# Locally:
DATA_DIR = '/kaggle/input/datasets/mehmetisik/hotel-intelligence-platform-data/'

transactions = pd.read_csv(f'{DATA_DIR}transactions.csv', parse_dates=['transaction_date'])
customers    = pd.read_csv(f'{DATA_DIR}customer_profiles.csv')

print(f'Transactions : {transactions.shape[0]:,} rows x {transactions.shape[1]} cols')
print(f'Customers    : {customers.shape[0]:,} rows x {customers.shape[1]} cols')
transactions.head(3)

Transactions : 52,979 rows x 11 cols
Customers    : 12,000 rows x 6 cols


,transaction_id,customer_id,transaction_date,nights,room_type,daily_rate,total_revenue,hotel_type,booking_channel,meal_plan,extra_spend
0,T000001,C00001,2024-02-22,2,Standard,980.93,1961.85,Resort Hotel,OTA,BB,61.82
1,T000002,C00001,2024-04-10,14,Superior,932.67,13057.32,Resort Hotel,OTA,RO,91.49
2,T000003,C00001,2024-04-24,3,Standard,806.77,2420.31,City Hotel,Travel Agent,HB,128.63


In [4]:
customers.head(3)

,customer_id,segment_true,expected_frequency,monetary_mean,country,customer_type
0,C00001,champion,12.49,770.43,DZ,Group
1,C00002,champion,15.18,293.61,CG,Business
2,C00003,champion,8.70,719.71,AR,Leisure


In [5]:
# Quick sanity checks
print('Transaction date range :', transactions['transaction_date'].min().date(),
      'to', transactions['transaction_date'].max().date())
print('Unique customers (txn) :', transactions['customer_id'].nunique())
print('Missing values (txn)   :')
print(transactions.isnull().sum()[transactions.isnull().sum() > 0])
print()
transactions.describe()

Transaction date range : 2024-01-01 to 2026-06-25
Unique customers (txn) : 12000
Missing values (txn)   :
Series([], dtype: int64)



,transaction_date,nights,daily_rate,total_revenue,extra_spend
count,52979,52979.00,52979.00,52979.00,52979.00
mean,2025-01-30 12:54:14.330961408,4.26,460.82,1956.03,52.15
min,2024-01-01 00:00:00,1.00,30.00,30.00,0.00
25%,2024-07-27 00:00:00,2.00,212.31,610.51,23.11
50%,2025-01-19 00:00:00,3.00,350.64,1189.39,50.15
75%,2025-08-07 00:00:00,5.00,584.01,2337.53,77.03
max,2026-06-25 00:00:00,14.00,5024.91,51734.46,210.24
std,NaN,3.11,383.47,2443.68,36.42


---
## 4. RFM Analysis

RFM segments customers along three dimensions:

| Metric | Meaning | Good direction |
|--------|---------|----------------|
| **Recency** | Days since last transaction | Lower is better |
| **Frequency** | Number of transactions | Higher is better |
| **Monetary** | Total revenue generated | Higher is better |

In [6]:
# Reference date = one day after last transaction
reference_date = transactions['transaction_date'].max() + timedelta(days=1)
print('Reference date:', reference_date.date())

rfm = (
    transactions
    .groupby('customer_id')
    .agg(
        recency   = ('transaction_date', lambda x: (reference_date - x.max()).days),
        frequency = ('transaction_id', 'nunique'),
        monetary  = ('total_revenue', 'sum')
    )
    .reset_index()
)

print(f'RFM table: {rfm.shape[0]:,} customers')
rfm.describe()

Reference date: 2026-06-26
RFM table: 12,000 customers


,recency,frequency,monetary
count,12000.00,12000.00,12000.00
mean,424.14,4.41,8635.70
std,270.80,4.49,15078.93
min,1.00,1.00,30.00
25%,186.00,1.00,821.77
50%,323.00,3.00,2652.01
75%,677.00,6.00,9508.40
max,907.00,32.00,163437.60


### 4.1 Quintile Scoring (1-5)

Each metric is binned into quintiles. For **Recency**, rank 5 means
the *most recent* (best); for Frequency & Monetary, rank 5 means the highest.

In [7]:
# Recency: lower days = better = higher score
rfm['R'] = pd.qcut(rfm['recency'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)

# Frequency & Monetary: higher = better = higher score
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M'] = pd.qcut(rfm['monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm['RFM_Score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)
rfm['RFM_Total'] = rfm['R'] + rfm['F'] + rfm['M']

rfm.head(10)

,customer_id,recency,frequency,monetary,R,F,M,RFM_Score,RFM_Total
0,C00001,144,17,88729.98,5,5,5,555,15
1,C00002,198,14,18103.36,4,5,5,455,14
2,C00003,208,15,40322.29,4,5,5,455,14
3,C00004,190,23,40155.13,4,5,5,455,14
4,C00005,180,22,48242.01,4,5,5,455,14
5,C00006,264,15,37300.63,3,5,5,355,13
6,C00007,231,7,21218.49,4,4,5,445,13
7,C00008,234,8,17547.15,4,5,5,455,14
8,C00009,172,21,100580.93,4,5,5,455,14
9,C00010,162,20,91192.32,5,5,5,555,15


### 4.2 Segment Assignment

We map RFM scores to business-meaningful labels using a rule-based approach.

In [8]:
def assign_segment(row):
    """Rule-based RFM segment assignment."""
    r, f, m = row['R'], row['F'], row['M']

    if r >= 4 and f >= 4 and m >= 4:
        return 'Champion'
    elif r >= 3 and f >= 3:
        return 'Loyal'
    elif r >= 4 and f <= 2:
        return 'New Customer'
    elif r >= 3 and m >= 3:
        return 'Potential'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    else:
        return 'Lost'

rfm['segment'] = rfm.apply(assign_segment, axis=1)

seg_counts = rfm['segment'].value_counts()
print('Segment distribution:')
print(seg_counts)
print(f'\nTotal customers: {seg_counts.sum():,}')

Segment distribution:
segment
Lost            4207
Loyal           3237
Champion        3082
At Risk          881
New Customer     387
Potential        206
Name: count, dtype: int64

Total customers: 12,000


### 4.3 Segment Distribution (Pie Chart)

In [9]:
color_map = {
    'Champion':    '#2ecc71',
    'Loyal':       '#3498db',
    'New Customer':'#9b59b6',
    'Potential':   '#f1c40f',
    'At Risk':     '#e67e22',
    'Lost':        '#e74c3c'
}

fig = px.pie(
    values=seg_counts.values,
    names=seg_counts.index,
    title='RFM Segment Distribution',
    color=seg_counts.index,
    color_discrete_map=color_map,
    hole=0.35
)
fig.update_traces(textinfo='percent+label', pull=[0.05]*len(seg_counts))
fig.update_layout(width=700, height=500)
fig.show()

### 4.4 RFM Scatter Plot (Recency vs Monetary)

In [10]:
fig = px.scatter(
    rfm,
    x='recency',
    y='monetary',
    size='frequency',
    color='segment',
    color_discrete_map=color_map,
    hover_data=['customer_id', 'RFM_Score'],
    title='RFM Scatter -- Recency vs Monetary (size = Frequency)',
    labels={'recency': 'Recency (days)', 'monetary': 'Total Revenue ($)'},
    opacity=0.65,
    size_max=20
)
fig.update_layout(width=900, height=600)
fig.show()

### 4.5 RFM Heatmap -- Average Monetary by R & F Scores

In [11]:
heatmap_data = rfm.pivot_table(index='F', columns='R', values='monetary', aggfunc='mean')

fig = px.imshow(
    heatmap_data,
    text_auto='.0f',
    color_continuous_scale='YlOrRd',
    labels={'x': 'Recency Score', 'y': 'Frequency Score', 'color': 'Avg Revenue ($)'},
    title='Average Monetary Value by Recency & Frequency Scores'
)
fig.update_layout(width=700, height=500)
fig.show()

---
## 5. CLTV Modeling (BG-NBD + Gamma-Gamma)

We use probabilistic models from the **lifetimes** library:

- **BG-NBD** models the *number* of future purchases a customer will make.
- **Gamma-Gamma** models the *monetary value* per transaction.
- Combined, they produce a 6-month Customer Lifetime Value prediction.

In [12]:
# Build the summary table required by lifetimes
cltv_input = summary_data_from_transaction_data(
    transactions,
    customer_id_col='customer_id',
    datetime_col='transaction_date',
    monetary_value_col='total_revenue',
    observation_period_end=reference_date
)

# Filter customers with at least 2 transactions (needed for Gamma-Gamma)
cltv_input = cltv_input[cltv_input['frequency'] > 0]
print(f'Customers eligible for CLTV modeling: {cltv_input.shape[0]:,}')
cltv_input.head()

Customers eligible for CLTV modeling: 7,993


,frequency,recency,T,monetary_value
customer_id,,,,
C00001,16.00,711.00,855.00,5423.01
C00002,13.00,616.00,814.00,1200.57
C00003,14.00,695.00,903.00,2721.90
C00004,22.00,667.00,857.00,1805.72
C00005,20.00,649.00,829.00,2296.24


### 5.1 BG-NBD Model -- Purchase Frequency

In [13]:
bgf = BetaGeoFitter(penalizer_coef=0.01)
bgf.fit(
    cltv_input['frequency'],
    cltv_input['recency'],
    cltv_input['T']
)
print('BG-NBD Model Parameters:')
print(bgf.summary)

# Predict expected purchases in the next 6 months (180 days)
cltv_input['exp_purchases_6m'] = bgf.predict(
    180,
    cltv_input['frequency'],
    cltv_input['recency'],
    cltv_input['T']
)

print(f'\nAverage expected purchases (6 months): {cltv_input["exp_purchases_6m"].mean():.2f}')

BG-NBD Model Parameters:
        coef  se(coef)  lower 95% bound  upper 95% bound
r       2.12      0.04             2.04             2.19
alpha 238.41      4.93           228.75           248.08
a       0.24      0.01             0.22             0.26
b       1.37      0.05             1.27             1.47

Average expected purchases (6 months): 0.88


### 5.2 Gamma-Gamma Model -- Monetary Value

In [14]:
ggf = GammaGammaFitter(penalizer_coef=0.01)
ggf.fit(
    cltv_input['frequency'],
    cltv_input['monetary_value']
)
print('Gamma-Gamma Model Parameters:')
print(ggf.summary)

cltv_input['exp_avg_revenue'] = ggf.conditional_expected_average_profit(
    cltv_input['frequency'],
    cltv_input['monetary_value']
)

print(f'\nAverage expected revenue per transaction: ${cltv_input["exp_avg_revenue"].mean():,.2f}')

Gamma-Gamma Model Parameters:
   coef  se(coef)  lower 95% bound  upper 95% bound
p  3.31      0.06             3.20             3.43
q  0.22      0.00             0.22             0.23
v  3.23      0.06             3.11             3.34

Average expected revenue per transaction: $1,779.91


### 5.3 Six-Month CLTV Prediction

In [15]:
cltv_input['cltv_6m'] = ggf.customer_lifetime_value(
    bgf,
    cltv_input['frequency'],
    cltv_input['recency'],
    cltv_input['T'],
    cltv_input['monetary_value'],
    time=6,        # months
    freq='D',      # daily frequency in input
    discount_rate=0.01
)

print('CLTV 6-month distribution:')
cltv_input['cltv_6m'].describe()

CLTV 6-month distribution:


count    7993.00
mean     1742.51
std      2055.79
min         3.75
25%       380.78
50%      1014.24
75%      2338.14
max     19169.68
Name: cltv_6m, dtype: float64

In [16]:
# Assign CLTV segments
cltv_input['cltv_segment'] = pd.qcut(
    cltv_input['cltv_6m'], q=4,
    labels=['Low', 'Medium', 'High', 'VIP']
)

cltv_seg = cltv_input['cltv_segment'].value_counts().sort_index()
print('CLTV Segment Counts:')
print(cltv_seg)

CLTV Segment Counts:
cltv_segment
Low       1999
Medium    1998
High      1998
VIP       1998
Name: count, dtype: int64


In [17]:
# Top 10 most valuable customers
top10 = (
    cltv_input
    .nlargest(10, 'cltv_6m')
    [['frequency', 'recency', 'T', 'monetary_value',
      'exp_purchases_6m', 'exp_avg_revenue', 'cltv_6m', 'cltv_segment']]
)
top10.style.format({
    'monetary_value': '${:,.0f}',
    'exp_avg_revenue': '${:,.0f}',
    'cltv_6m': '${:,.0f}',
    'exp_purchases_6m': '{:.1f}'
})

,frequency,recency,T,monetary_value,exp_purchases_6m,exp_avg_revenue,cltv_6m,cltv_segment
customer_id,,,,,,,,
C00129,20.000000,628.000000,753.000000,"$6,168",3.2,"$6,241","$19,170",VIP
C00821,13.000000,655.000000,849.000000,"$9,548",1.8,"$9,724","$16,990",VIP
C00857,18.000000,691.000000,840.000000,"$6,417",2.6,"$6,502","$16,350",VIP
C00702,14.000000,708.000000,844.000000,"$7,154",2.3,"$7,276","$16,081",VIP
C03539,1.000000,291.000000,403.000000,"$18,990",0.6,"$24,834","$15,122",VIP
C00625,21.000000,639.000000,801.000000,"$6,194",2.5,"$6,264","$15,088",VIP
C00012,22.000000,682.000000,844.000000,"$5,873",2.5,"$5,937","$14,609",VIP
C00149,22.000000,713.000000,855.000000,"$5,008",3.0,"$5,062","$14,508",VIP
C00343,14.000000,601.000000,747.000000,"$6,236",2.4,"$6,343","$14,400",VIP


In [18]:
# CLTV distribution by segment
cltv_colors = {'Low': '#e74c3c', 'Medium': '#f39c12', 'High': '#3498db', 'VIP': '#2ecc71'}

fig = px.box(
    cltv_input,
    x='cltv_segment',
    y='cltv_6m',
    color='cltv_segment',
    color_discrete_map=cltv_colors,
    title='6-Month CLTV Distribution by Segment',
    labels={'cltv_6m': 'CLTV ($)', 'cltv_segment': 'Segment'},
    category_orders={'cltv_segment': ['Low', 'Medium', 'High', 'VIP']}
)
fig.update_layout(width=800, height=500, showlegend=False)
fig.show()

In [19]:
# Expected purchases vs CLTV scatter
fig = px.scatter(
    cltv_input,
    x='exp_purchases_6m',
    y='cltv_6m',
    color='cltv_segment',
    color_discrete_map=cltv_colors,
    opacity=0.5,
    title='Expected Purchases vs 6-Month CLTV',
    labels={'exp_purchases_6m': 'Expected Purchases (6 months)',
            'cltv_6m': 'CLTV ($)'},
    category_orders={'cltv_segment': ['Low', 'Medium', 'High', 'VIP']}
)
fig.update_layout(width=900, height=550)
fig.show()

---
## 6. K-Means Behavioral Clustering

While RFM and CLTV use domain-specific rules and probabilistic models,
K-Means discovers **natural behavioral groups** in the data without prior assumptions.

In [20]:
# Feature engineering for clustering
cluster_features = (
    transactions
    .groupby('customer_id')
    .agg(
        total_spend      = ('total_revenue', 'sum'),
        avg_spend        = ('total_revenue', 'mean'),
        num_transactions = ('transaction_id', 'nunique'),
        avg_nights       = ('nights', 'mean'),
        avg_daily_rate   = ('daily_rate', 'mean'),
        avg_extra_spend  = ('extra_spend', 'mean')
    )
    .reset_index()
)

print(f'Clustering features for {cluster_features.shape[0]:,} customers')
cluster_features.describe()

Clustering features for 12,000 customers


,total_spend,avg_spend,num_transactions,avg_nights,avg_daily_rate,avg_extra_spend
count,12000.00,12000.00,12000.00,12000.00,12000.00,12000.00
mean,8635.70,1367.37,4.41,4.25,321.99,52.10
std,15078.93,1156.69,4.49,2.21,224.94,25.44
min,30.00,30.00,1.00,1.00,30.00,0.00
25%,821.77,550.39,1.00,3.00,163.87,37.08
50%,2652.01,1041.08,3.00,4.00,262.13,51.62
75%,9508.40,1847.37,6.00,5.00,412.11,65.60
max,163437.60,11464.18,32.00,14.00,1693.38,210.24


In [21]:
# Standardize features
feature_cols = ['total_spend', 'avg_spend', 'num_transactions',
                'avg_nights', 'avg_daily_rate', 'avg_extra_spend']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_features[feature_cols])
print('Scaled feature matrix shape:', X_scaled.shape)

Scaled feature matrix shape: (12000, 6)


### 6.1 Elbow Method + Silhouette Score

In [22]:
K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
    print(f'K={k:2d}  |  Inertia={km.inertia_:>12,.0f}  |  Silhouette={silhouettes[-1]:.4f}')

optimal_k = list(K_range)[np.argmax(silhouettes)]
print(f'\nBest K by Silhouette Score: {optimal_k}')

K= 2  |  Inertia=      47,162  |  Silhouette=0.4295
K= 3  |  Inertia=      38,775  |  Silhouette=0.2717
K= 4  |  Inertia=      32,359  |  Silhouette=0.3089
K= 5  |  Inertia=      26,643  |  Silhouette=0.2707
K= 6  |  Inertia=      24,258  |  Silhouette=0.2585
K= 7  |  Inertia=      22,374  |  Silhouette=0.2513
K= 8  |  Inertia=      20,646  |  Silhouette=0.2594
K= 9  |  Inertia=      19,016  |  Silhouette=0.2374
K=10  |  Inertia=      17,860  |  Silhouette=0.2335

Best K by Silhouette Score: 2


In [23]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Elbow Method (Inertia)', 'Silhouette Score'))

fig.add_trace(
    go.Scatter(x=list(K_range), y=inertias, mode='lines+markers',
               marker=dict(size=8, color='#3498db'), name='Inertia'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=list(K_range), y=silhouettes, mode='lines+markers',
               marker=dict(size=8, color='#e74c3c'), name='Silhouette'),
    row=1, col=2
)

# Highlight optimal K
fig.add_vline(x=optimal_k, line_dash='dash', line_color='green', row=1, col=1)
fig.add_vline(x=optimal_k, line_dash='dash', line_color='green', row=1, col=2)

fig.update_xaxes(title_text='Number of Clusters (K)', dtick=1)
fig.update_layout(width=1000, height=400, showlegend=False,
                  title_text=f'Optimal K = {optimal_k}')
fig.show()

### 6.2 Final Clustering & Profiling

In [24]:
kmeans_final = KMeans(n_clusters=optimal_k, n_init=20, random_state=42)
cluster_features['cluster'] = kmeans_final.fit_predict(X_scaled)

# Human-readable cluster labels will be assigned after profiling
cluster_profile = (
    cluster_features
    .groupby('cluster')[feature_cols]
    .mean()
    .round(2)
)

cluster_sizes = cluster_features['cluster'].value_counts().sort_index()
cluster_profile['count'] = cluster_sizes.values

print('Cluster Profiles (mean values):')
cluster_profile

Cluster Profiles (mean values):


,total_spend,avg_spend,num_transactions,avg_nights,avg_daily_rate,avg_extra_spend,count
cluster,,,,,,,
0,3176.50,974.40,2.88,4.13,243.49,52.03,9709
1,31771.16,3032.77,10.93,4.74,654.70,52.40,2291


In [25]:
# Assign human-readable names based on profile
# Sort clusters by total_spend to create a ranking
spend_rank = cluster_profile['total_spend'].rank(ascending=True).astype(int)

name_map_options = {
    1: 'Budget Travelers',
    2: 'Casual Guests',
    3: 'Regular Guests',
    4: 'Frequent Guests',
    5: 'Premium Guests',
    6: 'VIP Guests',
    7: 'Ultra Premium',
    8: 'Elite',
    9: 'Platinum',
}

cluster_name_map = {cluster_id: name_map_options.get(rank, f'Cluster {rank}')
                    for cluster_id, rank in spend_rank.items()}

cluster_features['cluster_name'] = cluster_features['cluster'].map(cluster_name_map)
cluster_profile.index = cluster_profile.index.map(cluster_name_map)

print('Cluster Name Mapping:', cluster_name_map)
print()
cluster_profile

Cluster Name Mapping: {0: 'Budget Travelers', 1: 'Casual Guests'}



,total_spend,avg_spend,num_transactions,avg_nights,avg_daily_rate,avg_extra_spend,count
cluster,,,,,,,
Budget Travelers,3176.50,974.40,2.88,4.13,243.49,52.03,9709
Casual Guests,31771.16,3032.77,10.93,4.74,654.70,52.40,2291


### 6.3 Radar Chart -- Cluster Comparison

In [26]:
# Normalize each feature to 0-1 for the radar chart
radar_df = cluster_profile[feature_cols].copy()
for col in feature_cols:
    min_val = radar_df[col].min()
    max_val = radar_df[col].max()
    if max_val > min_val:
        radar_df[col] = (radar_df[col] - min_val) / (max_val - min_val)
    else:
        radar_df[col] = 0.5

radar_colors = px.colors.qualitative.Set2

fig = go.Figure()
for i, (cluster_name, row) in enumerate(radar_df.iterrows()):
    fig.add_trace(go.Scatterpolar(
        r=row.values.tolist() + [row.values[0]],  # close the polygon
        theta=feature_cols + [feature_cols[0]],
        fill='toself',
        name=cluster_name,
        opacity=0.6,
        line=dict(color=radar_colors[i % len(radar_colors)])
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Cluster Radar Profile Comparison',
    width=800,
    height=600
)
fig.show()

In [27]:
# Cluster size bar chart
fig = px.bar(
    x=list(cluster_name_map.values()),
    y=cluster_sizes.values,
    color=list(cluster_name_map.values()),
    color_discrete_sequence=px.colors.qualitative.Set2,
    title='Cluster Size Distribution',
    labels={'x': 'Cluster', 'y': 'Number of Customers'},
    text=cluster_sizes.values
)
fig.update_traces(textposition='outside')
fig.update_layout(width=800, height=450, showlegend=False)
fig.show()

### 6.4 Cluster Scatter (2D PCA Projection)

In [28]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

cluster_features['pca1'] = X_pca[:, 0]
cluster_features['pca2'] = X_pca[:, 1]

fig = px.scatter(
    cluster_features,
    x='pca1', y='pca2',
    color='cluster_name',
    color_discrete_sequence=px.colors.qualitative.Set2,
    title=f'K-Means Clusters (PCA 2D Projection, explained var = {pca.explained_variance_ratio_.sum():.1%})',
    labels={'pca1': 'PC1', 'pca2': 'PC2', 'cluster_name': 'Cluster'},
    opacity=0.5
)
fig.update_layout(width=900, height=550)
fig.show()

---
## 7. Cross-Analysis: RFM Segments vs Clusters

In [29]:
# Merge RFM segments with cluster labels
cross = rfm[['customer_id', 'segment']].merge(
    cluster_features[['customer_id', 'cluster_name']],
    on='customer_id'
)

cross_tab = pd.crosstab(cross['segment'], cross['cluster_name'], normalize='index')

fig = px.imshow(
    cross_tab.round(2),
    text_auto='.0%',
    color_continuous_scale='Blues',
    title='RFM Segment vs Behavioral Cluster (Row-Normalized)',
    labels={'x': 'Cluster', 'y': 'RFM Segment', 'color': 'Proportion'}
)
fig.update_layout(width=800, height=500)
fig.show()

---
## 8. Business Recommendations

### RFM-Based Actions

| Segment | Strategy | Example Actions |
|---------|----------|----------------|
| **Champion** | Reward & retain | Exclusive loyalty perks, early access to new properties, personalized concierge |
| **Loyal** | Upsell & engage | Room upgrade offers, spa packages, anniversary recognition |
| **New Customer** | Onboard & convert | Welcome sequence, first-stay discount on return, feedback surveys |
| **Potential** | Nurture | Targeted email campaigns, seasonal promotions, loyalty program invitation |
| **At Risk** | Win back | Re-engagement emails, time-limited offers, satisfaction follow-up |
| **Lost** | Reactivate (selectively) | High-value lost guests get personal outreach; low-value get automated win-back |

### CLTV-Based Prioritization

- **VIP tier** customers drive disproportionate revenue. Assign dedicated relationship managers.
- **High tier** guests are the growth engine. Invest in converting them to VIP through personalization.
- **Medium tier** represents the bulk of the customer base. Optimize operational efficiency for this group.
- **Low tier** requires cost-effective automated engagement. Avoid over-investing in retention.

### Cluster-Driven Personalization

Each behavioral cluster should receive **tailored marketing content**:

- Budget travelers: Value-focused messaging, package deals, off-peak promotions.
- Premium/VIP guests: Luxury experience messaging, exclusive access, premium room categories.
- Extended-stay guests: Long-stay discounts, apartment-style accommodations, loyalty milestones.

## 9. Conclusion

This notebook demonstrated three complementary approaches to customer analytics on **7,993 eligible customers**:

1. **RFM Segmentation** provides a quick, interpretable view of customer health. High-frequency, high-recency customers (score 5,5) average **32,759 USD** in monetary value vs 804 USD for low-engagement guests.

2. **CLTV Modeling** (BG-NBD + Gamma-Gamma) quantifies the future value of each guest:
   - Average expected purchases (6 months): **0.88**
   - Average expected revenue per transaction: **1,779.91 USD**
   - Mean 6-month CLTV: **1,742.51 USD** (median: 1,014.24, max: 19,169.68)
   - 4 CLTV segments: Low, Medium, High, VIP (~2,000 customers each)

3. **K-Means Clustering** discovers natural behavioral groups that cut across RFM boundaries, revealing nuances like the distinction between high-frequency budget travelers and low-frequency premium guests.

**Key takeaways:**

- VIP customers (top quartile) have CLTV values reaching **19,169 USD** -- protecting these relationships is critical.
- The gap between Low (380 USD median) and VIP (5,000+ USD median) CLTV segments highlights the importance of targeted retention strategies.
- Combining RFM + CLTV + Clustering gives a **360-degree customer view** that no single method provides alone.

**Next steps:**

- Integrate these segments into the hotel CRM for automated campaign triggers.
- Build a real-time scoring API (see Module 4: MLOps) for operational deployment.
- Combine with review sentiment data (NLP pipeline) for a complete guest profile.

---
*Notebook by Mehmet Isik -- [Hotel Intelligence Platform](https://github.com/mmehmetisik/hotel-intelligence-platform)*


---

## 📚 Hotel Intelligence Platform -- Notebook Series

You've just segmented 7,993 customers and predicted their lifetime value. But what are they saying about your hotel?

| # | Notebook | What You'll Learn |
|---|----------|-------------------|
| 📊 | [01 - Cancellation Prediction](https://www.kaggle.com/code/mehmetisik/01-cancellation-prediction) | 5 ML models, SHAP explainability, EUR 13.9M+ impact |
| ✅ | **02 - Customer Analytics** | You are here |
| 👉 | [03 - NLP & LLM Pipeline](https://www.kaggle.com/code/mehmetisik/03-nlp-and-llm) | Invoice classification, item cleanup, sentiment analysis |
| 🏗️ | [04 - Platform Overview](https://www.kaggle.com/code/mehmetisik/04-platform-overview) | Full architecture and module integration |

🔗 [GitHub Repository](https://github.com/mmehmetisik/hotel-intelligence-platform)
